In [1]:
from typing import Any, SupportsFloat
import pygame
from pygame.locals import *
import pygame.base
from pkg_resources import resource_stream, resource_exists
import random
import numpy as np
import time
import keyboard
import copy
import numpy
import gymnasium as gym
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecEnv
from gymnasium.wrappers import TimeLimit
import cv2
from gymnasium.envs.registration import register


class SnakeEnv:

    def __init__(self, height, width) -> None:
        self.height = height
        self.width = width
        self.observationSpace = [
            [0 for _ in range(self.width)] for _ in range(self.height)]
        pass

    def getValue(self, y, x):
        return self.observationSpace[y][x]

    def setValue(self, y, x, value):
        if self.observationSpace[y][x] != value:
            self.observationSpace[y][x] = value

    def setNull(self, y, x):
        self.observationSpace[y][x] = 0

    def print(self):
        print()
        for y in range(len(self.observationSpace)):
            print(self.observationSpace[y])

    def getDesignationValue(self, value):
        result = []
        for y in range(len(self.observationSpace)):
            for x in range(len(self.observationSpace[y])):
                if self.observationSpace[y][x] == value:
                    result.append((y, x))
        return result

    def getEnvironmentalData(self):
        return self.observationSpace


class Wall:

    def __init__(self, snakeEnv: SnakeEnv) -> None:
        self.value = 1
        self.snakeEnv = snakeEnv
        self.walls = []
        self.generate()
        pass

    def generate(self):
        snakeEnv = self.snakeEnv
        for y in range(snakeEnv.height):
            for x in range(snakeEnv.width):
                if x == 0 or x == snakeEnv.width - 1 or \
                        y == 0 or y == snakeEnv.height - 1:
                    snakeEnv.setValue(y, x, self.value)
                    self.walls.append((y, x))

    def isWall(self, y, x):
        for i in range(len(self.walls)):
            vy, vx = self.walls[i]
            if vy == y and vx == x:
                return True
        return False


class Food:

    def __init__(self, snakeEnv: SnakeEnv, seed=None) -> None:
        self.value = 2
        self.snakeEnv = snakeEnv
        if seed == None:
            seed = int(time.time())
        random.seed(seed)
        self.foodCoordinates = ()
        self.generate()
        pass

    def generate(self, remove=False):
        if remove and len(self.foodCoordinates) == 2:
            y, x = self.foodCoordinates
            self.snakeEnv.setNull(y, x)
        emptyPosition = self.snakeEnv.getDesignationValue(0)
        lenEmptyPosition = len(emptyPosition)
        if lenEmptyPosition <= 0:
            return True
        if lenEmptyPosition == 1:
            y, x = emptyPosition[0]
        else:
            ri = random.randint(0, len(emptyPosition) - 1)
            y, x = emptyPosition[ri]
        self.foodCoordinates = (y, x)
        self.snakeEnv.setValue(y, x, self.value)
        return False

    def getFoodCoordinates(self):
        return self.foodCoordinates

    def isFood(self, y, x):
        vy, vx = self.foodCoordinates
        return vy == y and vx == x

    def rendering(self):
        if len(self.foodCoordinates) == 2:
            y, x = self.foodCoordinates
            self.snakeEnv.setValue(y, x, self.value)


class SnakeBody:

    UP = 1
    DOWN = 2
    LEFT = 3
    RIGHT = 4

    def __init__(self, snakeEnv: SnakeEnv) -> None:
        self.snakeEnv = snakeEnv
        self.head_value = 3
        self.body_value = 4
        # Length
        self.len = 1
        # Direction of movement
        self.movingDirection = 1
        self.body = [(int(snakeEnv.height / 2), int(snakeEnv.width / 2))]
        self.end = ()
        self.generate()
        pass

    def getBody(self):
        return self.body

    def getHead(self):
        return self.body[0]

    def length(self):
        return len(self.body)

    def setAction(self, action):
        if action == 0:
            return
        if (action % 2 == 0 and action - 1 == self.movingDirection) or \
                (action % 1 == 0 and action + 1 == self.movingDirection):
            return
        self.movingDirection = action

    def lengthen(self):
        if len(self.end) == 2:
            y, x = self.end
            self.body.append((y, x))
            self.snakeEnv.setValue(y, x, self.body_value)
            self.end = ()

    def move(self):
        y, x = self.body[0]
        if self.movingDirection == SnakeBody.UP:
            y -= 1
        elif self.movingDirection == SnakeBody.DOWN:
            y += 1
        elif self.movingDirection == SnakeBody.LEFT:
            x -= 1
        elif self.movingDirection == SnakeBody.RIGHT:
            x += 1
        else:
            raise RuntimeError(f"Exception move : {self.movingDirection}")
        self.body.insert(0, (y, x))
        ey, ex = self.body.pop()
        self.end = (ey, ex)
        self.snakeEnv.setNull(ey, ex)
        self.generate()
        return (y, x)

    def headBodyStack(self):
        hy, hx = self.body[0]
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            if hy == y and hx == x:
                return True
        return False

    def isBody(self, hy, hx):
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            if hy == y and hx == x:
                return True
        return False

    def generate(self):
        y, x = self.body[0]
        self.snakeEnv.setValue(y, x, self.head_value)
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            self.snakeEnv.setValue(y, x, self.body_value)


class TcsV2Env(gym.Env):

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, **kwargs) -> None:
        self.image_size = kwargs.get("image_size", 10)
        self.show = False
        self.height = kwargs.get("height", 50)
        self.width = kwargs.get("width", 50)
        self.windowcolor = [(0, 0, 0), (255, 255, 255), (255, 0, 0),
                            (0, 0, 255), None]
        self.WINDOW_WIDTH = self.width * self.image_size
        self.WINDOW_HEIGHT = self.height * self.image_size
        self.WINDOW_BLACK = (0, 0, 0)
        self.obs_type = kwargs.get("obs_type", "rgb")
        self.outputType = kwargs.get("outputType", "Discrete")
        if self.outputType == "Discrete":
            self.action_space = gym.spaces.Discrete(5)
        elif self.outputType == "Box":
            self.action_space = gym.spaces.Box(
                low=0, high=4, shape=(1,), dtype=np.uint8
            )
        if self.obs_type == "rgb":
            self.observation_space = gym.spaces.Box(
                low=0, high=255, shape=(self.height * self.image_size, self.width * self.image_size, 3), dtype=np.uint8)
        if self.obs_type == "data":
            self.observation_space = gym.spaces.Box(
                low=0, high=255, shape=(5,), dtype=np.uint8)
        self.render_mode = kwargs.get("render_mode", None)
        self.mp4 = kwargs.get("mp4", False)
        self.mp4fps = kwargs.get("mp4fps", 30)
        assert self.render_mode is None or self.render_mode in self.metadata["render_modes"]
        super().__init__()

    def get_action_meanings(self):
        return ["NOOP", "UP", "DOWN", "LEFT", "RIGHT"]

    def step(self, action: Any) -> tuple[Any, SupportsFloat, bool, bool, dict[str, Any]]:
        if isinstance(action, np.ndarray):
            action = action[0]
            action = round(action)
        info = {}
        reward = 0
        self.snakebody.setAction(action)
        y, x = self.snakebody.move()
        done = False
        if self.snakebody.isBody(y, x):
            done = True
            info["message"] = "Come into contact with the body"
        if self.wall.isWall(y, x):
            done = True
            info["message"] = "hit the wall"
        if self.food.isFood(y, x):
            self.snakebody.lengthen()
            if self.food.generate():
                done = True
                vn_step = 1000 - self.num_step
                if vn_step < 0:
                    vn_step = 0
                reward += 100 * (vn_step / 1000)
                info["message"] = "Damn, the screen is completely covered with it"
                if self.show:
                    time.sleep(3)
            reward += 1
            # self.distanceFood = 20
        # else:
        #     fy, fx = self.food.getFoodCoordinates()
        #     jy = abs(fy - y)
        #     jx = abs(fx - x)
        #     distance = jy + jx
        #     if distance < self.distanceFood:
        #         self.distanceFood = distance
        #         reward += 1 - ((1 / 20) * distance)
        if reward == 0:
            reward -= 0.01
        else:
            fy, fx = self.food.getFoodCoordinates()
            distance = abs(fy - y) + abs(fx - x)
            reward -= 0.001 * distance  # bigger penalty the further away it is
            
        self.score += reward
        observation = self.environmentalData()

        if self.mp4 and self.obs_type != "rgb":
            self.render()

        self.num_step += 1
        return observation, reward, done, False, info

    def reset(self, *, seed: int | None = None, options: dict[str, Any] | None = None) -> tuple[Any, dict[str, Any]]:
        self.snakeEnv = SnakeEnv(self.height, self.width)
        self.wall = Wall(self.snakeEnv)
        self.snakebody = SnakeBody(self.snakeEnv)
        self.food = Food(self.snakeEnv, seed=seed)
        observation = self.environmentalData()
        self.score = 0
        self.num_step = 0
        # self.distanceFood = 20
        return observation, {}


    def environmentalData(self):
        if self.obs_type == "rgb":
            observation = self.render()
            return observation

        elif self.obs_type == "data":
            hy, hx = self.snakebody.getHead()
            fy, fx = self.food.getFoodCoordinates()
            hl = self.snakebody.length()
            return np.array([hy, hx, fy, fx, hl], dtype=np.uint8)  # ← must be numpy array

        return {}

    def render(self):
        if (self.render_mode == "human") and ("window" not in self.__dict__ or self.window == None):
            pygame.init()
            self.window = pygame.display.set_mode(
                (self.width * 10, self.height * 10))
            pygame.display.set_caption("tcs")
            self.window.fill(self.WINDOW_BLACK)
            self.show = True
        # Create a canvas
        if "canvas" not in self.__dict__ or self.canvas == None:
            self.canvas = pygame.Surface(
                (self.WINDOW_WIDTH, self.WINDOW_HEIGHT))
        if self.show and ("canvasN" not in self.__dict__ or self.canvasN == None):
            self.canvasN = pygame.Surface(
                (self.width * 10, self.height * 10)
            )
        self.canvas.fill(self.WINDOW_BLACK)
        if self.show:
            self.canvasN.fill(self.WINDOW_BLACK)

        observation = self.snakeEnv.getEnvironmentalData()
        for y in range(len(observation)):
            for x in range(len(observation[y])):
                value = observation[y][x]
                if value == 0 or self.windowcolor[value] == None:
                    continue
                pygame.draw.rect(
                    self.canvas, self.windowcolor[value], (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
                if self.show:
                    pygame.draw.rect(
                        self.canvasN, self.windowcolor[value], (
                            x * 10, y * 10, 10, 10)
                    )
        body = self.snakebody.getBody()
        bodylen = len(body)
        if bodylen > 1:
            colosNm = round(105 / (bodylen - 1))
            dcolosNm = 150 + colosNm
            for i in range(bodylen - 2, 0, -1):
                y, x = body[i]
                vvColos = (0, dcolosNm, 0)
                pygame.draw.rect(
                    self.canvas, vvColos, (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
                if self.show:
                    pygame.draw.rect(
                        self.canvasN, vvColos, (
                            x * 10, y * 10, 10, 10)
                    )
                if dcolosNm >= 255:
                    continue
                dcolosNm += colosNm
                if dcolosNm > 255:
                    dcolosNm = 255
            y, x = body[-1]
            vbcolos = (232, 255, 1)
            pygame.draw.rect(
                self.canvas, vbcolos, (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
            if self.show:
                pygame.draw.rect(
                    self.canvasN, vbcolos, (
                        x * 10, y * 10, 10, 10)
                )

        result = np.transpose(
            np.array(pygame.surfarray.pixels3d(self.canvas)), axes=(1, 0, 2)
        )
        if self.render_mode == "human":
            show_image = np.transpose(
                np.array(pygame.surfarray.pixels3d(self.canvasN)), axes=(1, 0, 2)
            )
            self.window.blit(self.canvasN, self.canvasN.get_rect())
            # Cleanup Event
            pygame.event.pump()
            pygame.display.update()
            # pygame.display.flip()
            # event = pygame.event.poll()
            # # pygame.display.update()
            # if event.type == pygame.QUIT:
            #     pygame.quit()
            #     exit()
            if self.mp4:
                if "outMp4" not in self.__dict__ or self.outMp4 == None:
                    height, width, _ = show_image.shape
                    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                    self.outMp4 = cv2.VideoWriter(
                        "1.mp4", fourcc, self.mp4fps, (width, height))
                self.outMp4.write(show_image)

        return result

    def close(self):
        del self.snakebody
        del self.wall
        del self.food
        del self.snakeEnv
        del self.score
        if self.mp4 and hasattr(self, "outMp4") and self.outMp4 is not None:
            self.outMp4.release()
        return super().close()


class ConstructEnv:

    def __init__(self) -> None:
        gym.register(id="tcs-v2", entry_point=TcsV2Env, max_episode_steps=2000,
                     kwargs={})
        pass

    def getDummyVecEnvs(self, num=32, **kwargs) -> DummyVecEnv:
        return DummyVecEnv([lambda: self.getEnv(**kwargs) for _ in range(num)])

    def getSudoVecEnvs(self, num=6, **kwargs) -> SubprocVecEnv:
        return SubprocVecEnv([lambda: self.getEnv(**kwargs) for _ in range(num)])

    def getEnv(self, **kwargs) -> TimeLimit:
        return TimeLimit(gym.make("tcs-v2", **kwargs), max_episode_steps=5000)




register(
    id="tcs-v2", entry_point=TcsV2Env, max_episode_steps=2000,
                     kwargs={}
)

pygame 2.6.1 (SDL 2.28.4, Python 3.11.14)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
import numpy as np
import random
import wandb
from tqdm.notebook import trange
from stable_baselines3.common.base_class import BaseAlgorithm
from stable_baselines3.common.policies import BasePolicy


class QTable(BaseAlgorithm):
    def __init__( #epsilon parameters replaced with temperature for Boltzman policy
        self,
        env,
        learning_rate = 0.1,
        gamma = 0.99,
        temperature_start = 1.0,
        temperature_end = 0.1,
        temperature_decay = 0.999,
        verbose = 1,
        device = "cpu",
    ):
        super().__init__(
            policy = BasePolicy,
            env = env,
            learning_rate = learning_rate,
            verbose = verbose,
            support_multi_env = True,
            device = device,
        )
        self.gamma = gamma
        self.temperature = temperature_start
        self.temperature_start = temperature_start
        self.temperature_end = temperature_end
        self.temperature_decay = temperature_decay

        self._setup_model()

    def _setup_model(self):
        # Q-table: obs is (5,) uint8 vector
        # [hy, hx, fy, fx, hl]
        self.bins = 8  # buckets per dimension
        self.obs_dims = self.observation_space.shape[0]
        self.n_actions = self.action_space.n

        # Q-table shape: (bins, bins, ..., bins, n_actions)
        table_shape = [self.bins] * self.obs_dims + [self.n_actions]
        self.q_table = np.zeros(table_shape, dtype=np.float32)

        print("Starting training...")
        print(f"State space size: {self.obs_dims}, Action space size: {self.n_actions}")
        print(f"Initial Q-table shape: {self.q_table.shape}")
        print(f"Initial Q-table sample:\n{self.q_table[:5, :]}")
        print("-" * 50)

        # Extract grid dimensions from the environment
        height = self.env.get_attr("height")[0]  # get from first env in vec_env
        width  = self.env.get_attr("width")[0]
        max_length = height * width  # theoretical maximum snake length

        # Per-dimension bin edges matched to actual value ranges
        self.bin_edges = [
            np.linspace(0, height,      self.bins + 1)[1:-1],  # hy
            np.linspace(0, width,       self.bins + 1)[1:-1],  # hx
            np.linspace(0, height,      self.bins + 1)[1:-1],  # fy
            np.linspace(0, width,       self.bins + 1)[1:-1],  # fx
            np.linspace(1, max_length,  self.bins + 1)[1:-1],  # hl
        ]

    def _discretize(self, obs):
        """Convert continuous obs array (n_envs, obs_dim) → tuple of bin indices."""
        indices = []
        for dim in range(self.obs_dims):
            idx = np.digitize(obs[:, dim], self.bin_edges[dim])  # (n_envs,)
            indices.append(idx)
        return indices  # list of (n_envs,) arrays

    def _get_q(self, indices):
        """Lookup Q-values for a batch of discretized states → (n_envs, n_actions)."""
        return self.q_table[tuple(indices)]  # (n_envs, n_actions)

    def _select_action(self, obs): #_select_action replaced epsilon for boltzman
        indices = self._discretize(obs)
        q_vals  = self._get_q(indices)  # (n_envs, n_actions)

        # Softmax over Q-values scaled by temperature
        q_scaled = q_vals / self.temperature
        q_scaled -= q_scaled.max(axis=1, keepdims=True)  # numerical stability
        exp_q = np.exp(q_scaled)
        probs = exp_q / exp_q.sum(axis=1, keepdims=True)  # (n_envs, n_actions)

        # Sample action for each env according to probabilities
        actions = np.array([
            np.random.choice(self.n_actions, p=probs[i])
            for i in range(self.env.num_envs)
        ])
        return actions

    def _update(self, obs, actions, rewards, next_obs, dones):
        indices      = self._discretize(obs)
        next_indices = self._discretize(next_obs)

        current_q  = self.q_table[tuple(indices) + (actions,)] # (n_envs,)
        max_next_q = self._get_q(next_indices).max(axis=1) # (n_envs,)

        # Bellman update
        target_q   = rewards + self.gamma * max_next_q * (1 - dones)
        td_error   = target_q - current_q

        # Q-table update
        self.q_table[tuple(indices) + (actions,)] += self.learning_rate * td_error

    def learn(self, total_timesteps, callback=None, **kwargs):
        self._setup_learn(total_timesteps, callback)

        obs = self._last_obs
        episode = 0
        episode_rewards = np.zeros(self.env.num_envs)
        td_errors = []

        with trange(total_timesteps, desc="Q-Table Training", unit="step") as pbar:
            while self.num_timesteps < total_timesteps:

                actions  = self._select_action(obs)
                new_obs, rewards, dones, infos = self.env.step(actions)

                self._update(obs, actions, rewards, new_obs, dones.astype(np.float32))

                episode_rewards += rewards

                obs = new_obs
                self._last_obs = new_obs
                self.num_timesteps += self.env.num_envs
                pbar.update(self.env.num_envs)

                # Linear epsilon decay # self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
                if any(dones):
                    episode += 1
                    self.temperature = max(self.temperature_end, self.temperature * self.temperature_decay)

                    finished_rewards = episode_rewards[dones]
                    wandb.log({
                        "episode_reward": finished_rewards.mean(),
                        "episode": episode,
                        "temperature": self.temperature,
                        "timestep": self.num_timesteps,
                    })
                    episode_rewards[dones] = 0

                if self.num_timesteps % 20_000 == 0:
                    mean_r, val_std_r = evaluate(self, n_episodes=10)
                    pbar.set_postfix({
                        "Episode": episode,
                        "Temperature": f"{self.temperature:.3f}",
                        "Mean Reward": f"{mean_r:.3f}",
                        "Std: ": f"{np.std(rewards):.3f}"
                    })
                    wandb.log({
                        "temperature": self.temperature,
                        "episode": episode,
                        "timestep": self.num_timesteps,
                        "mean_r": mean_r,
                        "std_r": np.std(rewards),
                    })

        return self

    def save_table(self, path="q_table.npy"):
        np.save(path, self.q_table)
        print(f"Q-table saved to {path}")

    def load_table(self, path="q_table.npy"):
        self.q_table = np.load(path)
        print(f"Q-table loaded from {path}")

If you decide to work on wrapper then you need to combine em into 1 class 

In [3]:
import gymnasium as gym    

#no penalty - snake learns to survive
#too large a penalty - snake learns to die quickly
class RewardShapingWrapper(gym.Wrapper):
    def __init__(self, env, step_penalty=-0.01, hunger_cap=100):
        super().__init__(env)
        self.step_penalty = step_penalty
        self.hunger_cap = hunger_cap
        self.steps_since_food = 0

    def reset(self, **kwargs):
        self.steps_since_food = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)

        if reward >= 1:  # food eaten (your env gives +1 per food)
            self.steps_since_food = 0
        else:
            self.steps_since_food += 1

        if self.steps_since_food >= self.hunger_cap:
            terminated = True
            info["message"] = "Starved"
            self.steps_since_food = 0

        reward += self.step_penalty
        return obs, reward, terminated, truncated, info
    
    


In [4]:
import wandb
from dotenv import load_dotenv
import os

load_dotenv()
    # wandb_key = os.getenv("WANDB_KEY")
try:
    wandb.login(key="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")
    api = wandb.Api()
    viewer = api.viewer
    print(f"Logged in as: {viewer.username}")
except wandb.errors.AuthenticationError:
    print("Not logged in")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\asus\_netrc
wandb: Currently logged in as: danylo-zemskyi (danylo-zemskyi-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged in as: danylo-zemskyi


Below is a single run pipeline, but use sweeps which are defined below

**still read my comments**

In [5]:
# DUE: set up sweeps search of params of q-table

import numpy as np
import torch
from stable_baselines3.common.env_util import make_vec_env
import wandb

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

def evaluate(model, n_episodes=10):
    vec_env_evaluate = make_vec_env(
    "tcs-v2",
     #add more envs until cpu cannot handle it
    #cpu is the main bottleneck, since it has to sample buffer
    n_envs=1,
    env_kwargs={
        "height"  : 8,
        "width"   : 8,
        "obs_type": "data",
    },
    #wrapper_class=RewardShapingWrapper, #TODO: Consider uncommenting it in future runs
    # wrapper_kwargs={
    #     "step_penalty": -0.01,
    #     "closer_reward": 0.1,
    #     "further_penalty": -0.05,
    # }
)
    rewards = []
    for _ in range(n_episodes):
        obs = vec_env_evaluate.reset()
        total = 0
        done = [False]
        while not done[0]:
            #action = model.q_net(torch.tensor(obs, dtype=torch.float32).to(model.device)).argmax(dim=1).cpu().numpy()
            action = model._get_q(model._discretize(obs)).argmax(axis=1) # because in tabular, values are looked up in the table directly; there is no target nn
            obs, r, done, _ = vec_env_evaluate.step(action)
            total += r[0]
        rewards.append(total)
    vec_env_evaluate.close()
    return np.mean(rewards), np.std(rewards)

best_reward = -np.inf

def train():
    global best_reward
    run = wandb.init()
    config = run.config

    vec_env = make_vec_env(
        "tcs-v2",
         #add more envs until cpu cannot handle it
        #cpu is the main bottleneck, since it has to sample buffer
        n_envs=16,
        env_kwargs={
            "height"  : 8,
            "width"   : 8,
            "obs_type": "data",
        },
        #wrapper_class=RewardShapingWrapper, #TODO: Consider uncommenting it in future runs
        # wrapper_kwargs={
        #     "step_penalty": -0.01,
        #     "closer_reward": 0.1,
        #     "further_penalty": -0.05,
        # }
    )

    model = QTable(
        env = vec_env,
        learning_rate = config.learning_rate, #0.1,
        #batch_size  = config.batch_size,
        gamma = config.gamma, #0.99,
        temperature_start = 1.0,
        temperature_end = 0.1,
        temperature_decay = 0.999, #80_000 #Formerly, epsilon_decay_steps.Changed to "no steps" for dynamicity / smoother epsilon decay curve.
        #buffer_size = config.buffer_size,
        verbose = 1,
        device = DEVICE
    )
    model.learn(total_timesteps=500_000)
    mean_reward, std_reward = evaluate(model, n_episodes=50)
    wandb.log({"mean_reward": mean_reward, "std_reward": std_reward})

    # save only if best across entire sweep
    if mean_reward > best_reward:
        best_reward = mean_reward
        model_path  = "best_sweep_model.pt" #TRY NEXT MAYBE? "best_sweep_model.npy"
        torch.save({
            "q_table": model.q_table,
            "mean_reward": best_reward,
            "config": dict(config),
        }, model_path)
        wandb.save(model_path)

        # Mark this run as the best in the sweep
        run.summary["best_reward"] = best_reward
        run.summary["is_best"]     = True
        print(f"New sweep best: {best_reward:.2f} | Run: {run.id}")

    vec_env.close()
    run.finish()


Device: cuda


In [6]:
# Grid search set up with wandb

sweep_config = {
    "method": "bayes", #possible values random, grid, and bayes just as seach classes
    "metric": {
        "name": "mean_reward",
        "goal": "maximize"
    },
    "parameters": {
        "learning_rate": {
            "distribution": "uniform",
            "min": 0.05,
            "max": 0.5
            },
        "gamma": {
            "values": [0.995, 0.998, 0.999, 0.997]
            },
        "temperature_decay": {
            "values": [0.999, 0.995, 0.99]
            },
        "temperature_start": {
            "values": [1.0, 2.0, 5.0]
            }
    }
}


if __name__ == "__main__":
    sweep_id = wandb.sweep(sweep_config, project="Snake_DRL")
    wandb.agent(sweep_id, function=train, count=25) #TODO: Change to 50 when running experiments, 1 is for debugging


Create sweep with ID: ajb34ecs
Sweep URL: https://wandb.ai/danylo-zemskyi-city-university-of-london/Snake_DRL/sweeps/ajb34ecs


wandb: Agent Starting Run: sxvroqf8 with config:
wandb: 	gamma: 0.995
wandb: 	learning_rate: 0.14540756524054335
wandb: 	temperature_decay: 0.999
wandb: 	temperature_start: 2
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\asus\_netrc.


Using cuda device
Starting training...
State space size: 5, Action space size: 5
Initial Q-table shape: (8, 8, 8, 8, 8, 5)
Initial Q-table sample:
[[[[[[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    ...

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0

Q-Table Training:   0%|          | 0/500000 [00:00<?, ?step/s]

wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


New sweep best: 2.87 | Run: sxvroqf8


... (103 outputs left out)

wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: xwwrmk9a with config:
wandb: 	gamma: 0.998
wandb: 	learning_rate: 0.3689293835681708
wandb: 	temperature_decay: 0.99
wandb: 	temperature_start: 5
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\asus\_netrc.


Using cuda device
Starting training...
State space size: 5, Action space size: 5
Initial Q-table shape: (8, 8, 8, 8, 8, 5)
Initial Q-table sample:
[[[[[[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    ...

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0

Q-Table Training:   0%|          | 0/500000 [00:00<?, ?step/s]

episode,▁▁▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇█████
episode_reward,▁▁▁▂▃▄▃▁▃▄▄▂▁▁▃▄▃▄▆▃▂▂▁▂▄▆▄▄▆▅▄▃▃▇▅▇▇█▂▇
mean_r,▅▆▆▁█▇▆▇███▆▆▇██▆▆▆██▆█▇█
mean_reward,▁
std_r,▆█▇▅▅▃▆▃█▁▃▁▃▁█▁▆▃▆▃▁▁▁▅▃
std_reward,▁
temperature,█▇▆▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
timestep,▁▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
episode,14646
episode_reward,2.842
mean_r,6.3643


Eval pipeline to generate video and eval results

In [7]:
import torch as th
import numpy as np
from stable_baselines3.common.env_util import make_vec_env

def eval_after_reloading(model_being_evaluated):
    # Create vectorized environment
    vec_env = make_vec_env(
        "tcs-v2",
        env_kwargs={
            "height": 8,
            "width": 8,
            "obs_type": "data",
            "render_mode": "human",
            "mp4": True,
        },
        n_envs=1,
        #wrapper_class=RewardShapingWrapper #TODO: Consider uncommenting it in future runs
    )

    # Get dimensions
    obs_dim = vec_env.observation_space.shape[0]
    n_actions = vec_env.action_space.n

    if model_being_evaluated == "model_debug":
        # Load Q-table
        model_being_evaluated = QTable(env = vec_env,verbose = 0)
        model_being_evaluated.load_table("q_table.npy")

    elif model_being_evaluated: # == "model": # or "best_sweep_model.pt":
        checkpoint = torch.load("best_sweep_model.pt", weights_only=False) #checkpoint = torch.load("best_sweep_model.npy", weights_only=False)
        print(f"Best reward: {checkpoint['mean_reward']:.2f}")
        print(f"Best config: {checkpoint['config']}")
        model_being_evaluated = QTable(env=vec_env, verbose=0)
        model_being_evaluated.q_table = checkpoint["q_table"]



    n_episodes = 10
    results    = []

    for ep in range(n_episodes):
        obs = vec_env.reset()
        total_reward = 0
        done = [False]

        while not done[0]:
            # Q-table lookup
            indices = model_being_evaluated._discretize(obs)
            q_vals  = model_being_evaluated._get_q(indices)
            action  = q_vals.argmax(axis=1)

            obs, rewards, done, infos = vec_env.step(action)
            total_reward += rewards[0]

        results.append(total_reward)
        print(f"Episode {ep+1}: reward={total_reward:.2f}")

    print(f"\nMean reward across evaluation pisodes: {np.mean(results):.2f} | Std: {np.std(results):.2f}")
    vec_env.close()



if __name__ == "__main__":
    #eval_after_reloading("model_debug")
    eval_after_reloading("model")

Best reward: 8.35
Best config: {'gamma': 0.999, 'learning_rate': 0.2679230938521627, 'temperature_decay': 0.995, 'temperature_start': 2}
Starting training...
State space size: 5, Action space size: 5
Initial Q-table shape: (8, 8, 8, 8, 8, 5)
Initial Q-table sample:
[[[[[[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    ...

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     ...
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0.]]

    [[0. 0

In [8]:
import wandb
import numpy as np

def analyze_sweep(experiment_to_analyze):
    api = wandb.Api()
    #sweep = api.sweep("danylo-zemskyi-city-university-of-london/Snake_DRL/dummy")
    sweep = api.sweep(experiment_to_analyze)
    best_run = max(sweep.runs, key=lambda r: r.summary.get("best_reward", -np.inf))
    print(f"Best run: {best_run.id}")
    print(f"Reward: {best_run.summary['mean_reward']:.2f}")
    print(f"Config: {best_run.config}")
    return best_run.file("best_sweep_model.pt").download(replace=True)  # saves to current directory

#if __name__ == "__main__":
    #model_to_evaluate = analyze_sweep("danylo-zemskyi-city-university-of-london/Snake_DRL/dummy")
    #eval_after_reloading("model_to_evaluate")